In [1]:
from ingest import load_data
documents = load_data()

In [2]:
documents[0]

{'id': '2',
 'recipe_name': "Sarah's Homemade Applesauce",
 'total_time': '25 mins',
 'servings': '4',
 'ingredients': '4  apples - peeled, cored and chopped, ¾ cup water, ¼ cup white sugar, ½ teaspoon ground cinnamon',
 'directions': "Combine apples, water, sugar, and cinnamon in a saucepan; cover and cook over medium heat until apples are soft, about 15 to 20 minutes.\nAllow apple mixture to cool, then mash with a fork or potato masher until it is the consistency you like.\n\n\n\n\n\n\n\n\n\n\n\nPhoto by cookin'mama.\ncookin mama\n",
 'nutrition': 'Total Fat 0g 0%, Sodium 3mg 0%, Total Carbohydrate 32g 12%, Dietary Fiber 4g 13%, Total Sugars 27g, Protein 0g, Vitamin C 6mg 32%, Calcium 13mg 1%, Iron 0mg 1%, Potassium 150mg 3%'}

In [3]:
from pydantic import BaseModel

class Questions(BaseModel):
    questions: list[str]

In [81]:
data_gen_instructions = """
You emulate a user who is searching for quick recipes ideas. Questions should contain
information about their available time, available groceries, nutritional goals or dietary preferences 
which correspond to the observed recipe record. 
If user is searching by available groceries they don't need to match the ingredients 
in the recipe completely. 
Dietary and nutritional goals should be reflected as well, not as exact numbers, but rather 
as high or low daily usage values by user preference.
Formulate 2 questions this user might ask to get particular recipe suggested. 

The output should resemble how people ask questions
on the internet. Not too formal, not too short, not too long.
""".strip()

In [82]:
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv()
openai_client = OpenAI()

In [83]:
doc = documents[200]

In [84]:
import json
user_prompt = json.dumps(doc)

In [85]:
doc

{'id': '343',
 'recipe_name': 'Amazing Lemon Scones',
 'total_time': '55 mins',
 'servings': '10',
 'ingredients': '3 cups all-purpose flour, ⅓ cup white sugar, 1 ½ teaspoons baking powder, 1 ½ teaspoons baking soda, ⅓ teaspoon salt, ¾ cup cold butter, cut into pieces, 9 tablespoons milk, 3 tablespoons lemon juice, 2 ½ teaspoons lemon zest, 1 ½ teaspoons vinegar',
 'directions': "Preheat the oven to 350 degrees F (175 degrees C).\nMake the scones: Whisk flour, sugar, baking powder, baking soda, and salt together in a large bowl. Cut in cold butter with 2 knives or a pastry blender until mixture resembles coarse crumbs. Whisk milk, lemon juice, lemon zest, and vinegar in a small bowl; stir into flour mixture until dough is moistened.\nTurn dough out onto a lightly floured surface and knead briefly, 5 or 6 turns. Pat or roll dough out into a 1-inch-thick round. Cut into 10 wedge-shaped pieces; arrange 1 inch apart on a baking sheet.\nBake in the preheated oven until bottom edges are gold

In [86]:
user_prompt

'{"id": "343", "recipe_name": "Amazing Lemon Scones", "total_time": "55 mins", "servings": "10", "ingredients": "3 cups all-purpose flour, \\u2153 cup white sugar, 1 \\u00bd teaspoons baking powder, 1 \\u00bd teaspoons baking soda, \\u2153 teaspoon salt, \\u00be cup cold butter, cut into pieces, 9 tablespoons milk, 3 tablespoons lemon juice, 2 \\u00bd teaspoons lemon zest, 1 \\u00bd teaspoons vinegar", "directions": "Preheat the oven to 350 degrees F (175 degrees C).\\nMake the scones: Whisk flour, sugar, baking powder, baking soda, and salt together in a large bowl. Cut in cold butter with 2 knives or a pastry blender until mixture resembles coarse crumbs. Whisk milk, lemon juice, lemon zest, and vinegar in a small bowl; stir into flour mixture until dough is moistened.\\nTurn dough out onto a lightly floured surface and knead briefly, 5 or 6 turns. Pat or roll dough out into a 1-inch-thick round. Cut into 10 wedge-shaped pieces; arrange 1 inch apart on a baking sheet.\\nBake in the p

In [87]:
messages = [
    {"role": "developer", "content": data_gen_instructions},
    {"role": "user", "content": user_prompt}
]

In [88]:
response = openai_client.responses.parse(
    model="gpt-5.4-mini",
    input=messages,
    text_format=Questions
)

In [89]:
response.output_parsed.questions

['Got about an hour and have flour, sugar, butter, milk, and lemons at home — what’s a good easy lemon scone recipe I can make?',
 'I want a quick sweet bake that’s fine for a low-protein, higher-carb treat, and I’ve got lemon juice and zest, flour, and butter — any lemon scone ideas?']

In [90]:
from evaluation_utils import llm_structured

In [23]:
from evaluation_utils import llm_structured_retry

In [24]:
def generate_ground_truth(doc):
    user_prompt = json.dumps(doc)

    out, usage = llm_structured_retry(
        openai_client,
        data_gen_instructions,
        user_prompt,
        Questions
    )

    results = []

    for q in out.questions:
        results.append({
            "question": q,
            "document": doc["id"]
        })

    return results, usage

In [25]:
from concurrent.futures import ThreadPoolExecutor
from evaluation_utils import map_progress

In [26]:
with ThreadPoolExecutor(max_workers=6) as pool:
    results = map_progress(pool, documents, generate_ground_truth)

  0%|          | 0/598 [00:00<?, ?it/s]

In [27]:
ground_truth = []
usages = []

for records, usage in results:
    ground_truth.extend(records)
    usages.append(usage)

len(ground_truth)

1196

In [28]:
ground_truth[10]

{'question': 'I’ve got a bunch of apples and want something quick—what’s an easy warm dessert I can make in about 20 minutes?',
 'document': '14'}

In [30]:
from evaluation_utils import calc_total_price

calc_total_price(usages)

0.3355124999999997

In [31]:
import pandas as pd

In [32]:
df_ground_truth = pd.DataFrame(ground_truth)

In [33]:
df_ground_truth.to_csv("data/ground_truth.csv", index=False)